# Projeto 4 — Álgebra Linear Numérica
**FGV–EMAp | 2026/1**  
Bernardo Freitas Paulo da Costa  
*Monitores: Adriel Dias Faria dos Santos e José Thevez Gomes Guedes*

---

## Questão 1 — Cálculo dos Refletores de Householder

Um **refletor de Householder** é uma matriz ortogonal da forma
$$Q_v = I - \beta\, v v^*,$$
armazenado compactamente pelo par $(v, \beta)$. O objetivo desta questão é derivar as fórmulas para $v$ e $\beta$, analisar o condicionamento e implementar a construção em Julia com atenção à estabilidade numérica.

---
## Item (a) — Derivação de $v$ e $\beta$

**Enunciado:** Mostrar que, para $Q_v x = \|x\|\, e_1$, temos
$$v = x - \|x\|\, e_1, \qquad
\beta = \frac{1}{\|x\|(\|x\| - x_1)}
     = \frac{\|x\| + x_1}{\|x\|(\|x\|^2 - x_1^2)}
     = \frac{\|x\| + x_1}{\|x\| \cdot \|y\|^2},$$
onde $y = (x_2, \ldots, x_n)$.

### Solução

Queremos $Q_v x = \|x\|\,e_1$, ou seja:
$$(I - \beta v v^*) x = \|x\|\, e_1.$$

**Passo 1 — Escolha de $v$.**  
Um refletor de Householder reflete $x$ em $\|x\|e_1$; o hiperplano de reflexão é a bissetriz entre $x$ e $\|x\|e_1$. O vetor normal a esse hiperplano é:
$$v = x - \|x\|\, e_1.$$

**Passo 2 — Determinação de $\beta$.**  
Para que $(I - \beta v v^*) x = \|x\|\, e_1$, precisamos:
$$x - \beta (v^* x)\, v = \|x\|\, e_1.$$

Como $v = x - \|x\|\, e_1$, temos:
$$v^* x = (x - \|x\|\, e_1)^* x = \|x\|^2 - \|x\|\, x_1.$$

Substituindo:
$$x - \beta(\|x\|^2 - \|x\| x_1)\,v = \|x\|\, e_1 \implies \beta(\|x\|^2 - \|x\| x_1)\,v = v.$$

Portanto, desde que $v \neq 0$:
$$\boxed{\beta = \frac{1}{\|x\|^2 - \|x\| x_1} = \frac{1}{\|x\|(\|x\| - x_1)}}.$$

**Passo 3 — Formas equivalentes.**  
Multiplicando numerador e denominador por $(\|x\| + x_1)$:
$$\beta = \frac{\|x\| + x_1}{\|x\|(\|x\|^2 - x_1^2)}.$$

Como $\|x\|^2 - x_1^2 = x_2^2 + \cdots + x_n^2 = \|y\|^2$ (onde $y = (x_2, \ldots, x_n)$):
$$\boxed{\beta = \frac{\|x\| + x_1}{\|x\| \cdot \|y\|^2}}.$$

**Verificação de ortogonalidade de $Q_v$:**  
Note que $\|v\|^2 = v^* v = \|x\|^2 - 2\|x\| x_1 + \|x\|^2 = 2(\|x\|^2 - \|x\| x_1)$, então $\beta = 2/\|v\|^2$, que é exatamente a condição para que $Q_v = I - \beta vv^*$ seja ortogonal ($Q_v^* = Q_v^{-1} = Q_v$). $\blacksquare$

---
## Item (b) — Cancelamento Numérico

**Enunciado:** Verificar que, dependendo do sinal de $x_1$, uma das fórmulas para $\beta$ não está sujeita a cancelamento numérico.

### Solução

As duas formas candidatas para o denominador de $\beta$ são:
- **Forma 1:** $\|x\|(\|x\| - x_1)$
- **Forma 2:** $\|x\|(\|x\| + x_1)$ (no denominador da expressão $\beta = (\|x\|+x_1)/(\|x\|\|y\|^2)$, veja a relação com a Forma 1)

O problema é que $v_1 = x_1 - \|x\|$ pode sofrer **cancelamento catastrófico**:

| Sinal de $x_1$ | $v_1 = x_1 - \|x\|$ | Problema |
|:---:|:---:|:---:|
| $x_1 > 0$ | $x_1 - \|x\| \approx 0$ se $x \approx \|x\|e_1$ | **Cancelamento em $v_1$** |
| $x_1 < 0$ | $x_1 - \|x\| \approx 2x_1$, sem cancelamento | Seguro |

Para o escalar $\beta$, vejamos a fórmula $\beta = 1/(\|x\|(\|x\| - x_1))$:

- Se $x_1 > 0$: o termo $\|x\| - x_1 \approx 0$ quando $x \approx \|x\|e_1$, causando **cancelamento catastrófico** no denominador.
- Se $x_1 < 0$: $\|x\| - x_1 = \|x\| + |x_1| > \|x\|$, não há cancelamento.

A fórmula alternativa $\beta = (\|x\| + x_1)/(\|x\| \cdot \|y\|^2)$ funciona quando $x_1 > 0$ (pois $\|x\| + x_1 \geq \|x\|$), mas falha quando $x_1 < 0$.

**Estratégia numericamente estável:** escolher o sinal de $\|x\|$ **oposto** ao de $x_1$, ou seja, usar $\sigma = -\text{sign}(x_1)$ e definir:
$$v_1 = x_1 + \sigma\|x\|, \quad \beta = \frac{-\sigma}{\|x\| v_1} = \frac{1}{|v_1| \cdot \|x\|}, \quad Q_v x = -\sigma\|x\|\,e_1.$$

Com essa escolha, $v_1 = x_1 + \sigma \|x\|$ nunca é pequeno (pois $x_1$ e $\sigma\|x\|$ têm o mesmo sinal), eliminando o cancelamento.

> **Obs.:** O enunciado pede que refletimos em $+\|x\|e_1$ (diagonal positiva em $R$). Para $x_1 > 0$, isso equivale a usar a fórmula $\beta = (\|x\|+x_1)/(\|x\| \|y\|^2)$, que evita cancelamento neste caso.

---
## Item (c) — Implementação: `reflector(x)`

A função recebe um vetor $x$ e retorna $(v, \beta)$ tal que $Q_v x = \|x\|\,e_1$ com diagonal positiva (i.e., reflete sempre em $+\|x\|e_1$, forçando $v_1 \leq 0$).

In [ ]:
using LinearAlgebra

"""
    reflector(x)

Calcula o refletor de Householder (v, β) tal que Q_v * x = ||x|| * e₁,
onde Q_v = I - β * v * v'.

A fórmula para β é escolhida para evitar cancelamento numérico:
  - Se x₁ ≤ 0: usa β = 1 / (||x|| * (||x|| - x₁))          (sem cancelamento pois ||x|| - x₁ ≥ ||x||)
  - Se x₁ > 0: usa β = (||x|| + x₁) / (||x|| * ||y||²)     (sem cancelamento pois ||x|| + x₁ ≥ ||x||)

Retorna (v, β) com o mesmo tipo de x (Float32 ou Float64).
"""
function reflector(x::AbstractVector{T}) where T <: AbstractFloat
    n    = length(x)
    nrmx = norm(x)           # ||x||
    
    v = copy(x)              # v = x (cópia, para não modificar x)
    v[1] -= nrmx             # v₁ = x₁ - ||x||  →  v = x - ||x||e₁
    
    # Cálculo de β evitando cancelamento numérico:
    # v₁ = x₁ - ||x||. Se x₁ > 0, há cancelamento; usamos a identidade
    # (||x|| - x₁)(||x|| + x₁) = ||x||² - x₁² = ||y||²
    if x[1] > 0
        # β = (||x|| + x₁) / (||x|| * ||y||²)
        # equivalente a  β = -1 / (nrmx * v₁)  mas numericamente melhor
        nrmy2 = nrmx^2 - x[1]^2   # ||y||² = sum(x[2:end].^2)
        β = (nrmx + x[1]) / (nrmx * nrmy2)
    else
        # β = 1 / (||x|| * (||x|| - x₁)) = 1 / (nrmx * (-v₁))
        β = one(T) / (nrmx * (-v[1]))   # -v₁ = ||x|| - x₁ > 0
    end
    
    return v, β
end

---
## Item (d) — Condicionamento de $v$ em relação a $x$

**Enunciado:** Mostrar que $J = \partial v / \partial x = I - e_1 x^* / \|x\|$ e que o número de condicionamento absoluto de $v$ em relação a $x$ é $\leq 2$.

### Solução

**Derivada de $v$:**  
Temos $v(x) = x - \|x\|\,e_1$. A derivada de $\|x\| = (x^*x)^{1/2}$ em relação a $x$ é $x^*/\|x\|$. Portanto:
$$J = \frac{\partial v}{\partial x} = I - e_1 \cdot \frac{x^*}{\|x\|}.$$

**Número de condicionamento absoluto:**  
O número de condicionamento absoluto de $v$ em relação a $x$ é $\|J\|_2 = \sigma_{\max}(J)$.

Seja $\hat{x} = x/\|x\|$ o vetor unitário na direção de $x$. Então:
$$J = I - e_1 \hat{x}^*.$$

Isso é uma **perturbação de posto 1** da identidade. Para qualquer vetor $u$:
$$Ju = u - (\hat{x}^* u)\, e_1.$$

Calculamos $\|Ju\|^2$:
$$\|Ju\|^2 = \|u\|^2 - 2(\hat{x}^* u)\, e_1^* u + (\hat{x}^* u)^2 \|e_1\|^2
           = \|u\|^2 - 2(\hat{x}^* u) u_1 + (\hat{x}^* u)^2.$$

Por Cauchy-Schwarz, $|\hat{x}^* u| \leq \|u\|$, logo:
$$\|Ju\|^2 \leq \|u\|^2 + 2\|u\|^2 + \|u\|^2 = 4\|u\|^2,$$

mas isso daria $\|J\|_2 \leq 2$. Vamos ser mais cuidadosos.

Usando a norma do operador de posto 1: $\|I - e_1\hat{x}^*\|_2 \leq \|I\|_2 + \|e_1\hat{x}^*\|_2 = 1 + \|e_1\|_2\|\hat{x}\|_2 = 1 + 1 = 2$.

Pela desigualdade triangular:
$$\boxed{\|J\|_2 \leq \|I\|_2 + \|e_1 \hat{x}^*\|_2 = 1 + 1 = 2.}$$

Portanto, o **número de condicionamento absoluto** de $v$ em relação a $x$ é $\|J\|_2 \leq 2$, o que significa que $v$ é um mapeamento bem condicionado de $x$. $\blacksquare$

---
## Item (e) — Estabilidade: testes Float64 e Float32

Verificamos que `reflector(x)` retorna resultados no mesmo tipo de `x`, e testamos a precisão da reflexão $Q_v x \approx \|x\| e_1$ em ambos os tipos.

In [ ]:
# Função auxiliar para aplicar o refletor: Q_v * x = (I - β*v*v') * x
apply_reflector_vec(v, β, x) = x - β * (v' * x) * v

# --- Teste 1: vetor genérico ---
println("=== Vetor genérico ===")
x64 = [3.0, 4.0, 0.0]          # Float64
x32 = Float32[3.0, 4.0, 0.0]   # Float32

v64, β64 = reflector(x64)
v32, β32 = reflector(x32)

println("Float64: typeof(v) = ", typeof(v64), ", typeof(β) = ", typeof(β64))
println("Float32: typeof(v) = ", typeof(v32), ", typeof(β) = ", typeof(β32))

Qx64 = apply_reflector_vec(v64, β64, x64)
Qx32 = apply_reflector_vec(v32, β32, x32)

nrmx64 = norm(x64)
nrmx32 = norm(x32)

println("\nFloat64: Q_v*x = ", Qx64)
println("Esperado:        [", nrmx64, ", 0, 0]")
println("Erro (Float64):  ", norm(Qx64 - [nrmx64, 0, 0]))

println("\nFloat32: Q_v*x = ", Qx32)
println("Esperado:        [", nrmx32, ", 0, 0]")
println("Erro (Float32):  ", norm(Qx32 - [nrmx32, 0, 0]))

In [ ]:
# --- Teste 2: vetor quase alinhado com e₁ (caso delicado para cancelamento) ---
println("=== Vetor quase alinhado com e₁ (x₁ > 0, pequenas componentes) ===")
ε64 = 1e-8
ε32 = Float32(1e-4)  # Float32 tem menos precisão

x64_near = [1.0, ε64, ε64]
x32_near = Float32[1.0, ε32, ε32]

v64n, β64n = reflector(x64_near)
v32n, β32n = reflector(x32_near)

Qx64n = apply_reflector_vec(v64n, β64n, x64_near)
Qx32n = apply_reflector_vec(v32n, β32n, x32_near)

nrm64n = norm(x64_near)
nrm32n = norm(x32_near)

println("\nFloat64: Q_v*x = ", Qx64n)
println("Erro relativo (Float64): ", norm(Qx64n - [nrm64n, 0, 0]) / nrm64n)

println("\nFloat32: Q_v*x = ", Qx32n)
println("Erro relativo (Float32): ", norm(Qx32n - [nrm32n, 0, 0]) / nrm32n)

In [ ]:
# --- Teste 3: vetor com x₁ < 0 (fórmula alternativa) ---
println("=== Vetor com x₁ < 0 ===")
x64_neg = [-3.0, 4.0, 0.0]
x32_neg = Float32[-3.0, 4.0, 0.0]

v64_neg, β64_neg = reflector(x64_neg)
v32_neg, β32_neg = reflector(x32_neg)

Qx64_neg = apply_reflector_vec(v64_neg, β64_neg, x64_neg)
Qx32_neg = apply_reflector_vec(v32_neg, β32_neg, x32_neg)

nrm64_neg = norm(x64_neg)
nrm32_neg = norm(x32_neg)

println("\nFloat64: Q_v*x = ", Qx64_neg, " | esperado: [", nrm64_neg, ", 0, 0]")
println("Erro relativo (Float64): ", norm(Qx64_neg - [nrm64_neg, 0, 0]) / nrm64_neg)

println("\nFloat32: Q_v*x = ", Qx32_neg, " | esperado: [", nrm32_neg, ", 0, 0]")
println("Erro relativo (Float32): ", norm(Qx32_neg - [nrm32_neg, 0, 0]) / nrm32_neg)

---
## Item (f) — Função `calc_beta(v)` e comparação

Dados apenas $v = x - \|x\|e_1$, podemos recuperar $\beta$ sem precisar de $x$ explicitamente.

### Derivação

Sabemos que $\beta = 2/\|v\|^2$ (condição de ortogonalidade de $Q_v$). De fato:
$$\|v\|^2 = \|x - \|x\|e_1\|^2 = \|x\|^2 - 2\|x\|x_1 + \|x\|^2 = 2(\|x\|^2 - \|x\|x_1) = 2\|x\|(\|x\| - x_1).$$

Portanto:
$$\boxed{\beta = \frac{2}{\|v\|^2}}.$$

Esta fórmula é **sempre válida** (desde que $v \neq 0$) e não apresenta cancelamento numérico, pois $\|v\|^2 > 0$.

In [ ]:
"""
    calc_beta(v)

Calcula β = 2 / ||v||² a partir do vetor v do refletor de Householder.
Preserva o tipo de v (Float32 ou Float64).
"""
function calc_beta(v::AbstractVector{T}) where T <: AbstractFloat
    return 2 / (v' * v)   # β = 2 / ||v||²
end

In [ ]:
# Comparação entre reflector(x) e calc_beta(v) usando o v retornado
println("=== Comparação reflector(x) vs calc_beta(v) ===")

test_cases = [
    ("Genérico Float64",      [3.0, 4.0, 0.0]),
    ("Genérico Float32",      Float32[3.0, 4.0, 0.0]),
    ("Quase e₁ Float64",      [1.0, 1e-8, 1e-8]),
    ("Quase e₁ Float32",      Float32[1.0, 1f-4, 1f-4]),
    ("x₁ < 0, Float64",       [-3.0, 4.0, 0.0]),
    ("x₁ < 0, Float32",       Float32[-3.0, 4.0, 0.0]),
]

for (name, x) in test_cases
    v, β_from_x = reflector(x)
    β_from_v    = calc_beta(v)
    
    # Erros de reflexão com cada β
    err_x = norm(apply_reflector_vec(v, β_from_x, x) - [norm(x); zeros(eltype(x), length(x)-1)])
    err_v = norm(apply_reflector_vec(v, β_from_v, x) - [norm(x); zeros(eltype(x), length(x)-1)])
    
    println("\n[$name]")
    println("  β (de reflector):  $β_from_x  | erro ||Q_v*x - ||x||e₁|| = $err_x")
    println("  β (de calc_beta):  $β_from_v  | erro ||Q_v*x - ||x||e₁|| = $err_v")
end

**Observação:** Como esperado, os dois valores de $\beta$ são praticamente idênticos e os erros de reflexão são comparáveis. A fórmula $\beta = 2/\|v\|^2$ é simples e numericamente estável, pois evita qualquer cancelamento.

---
## Item (g) — Bônus: Mau condicionamento de $v_1$ e fórmula alternativa

### Análise do condicionamento de $v_1$

Temos $v_1 = x_1 - \|x\|$. Quando $x$ está quase alinhado com $e_1$ (i.e., $x \approx \|x\|e_1$, com $x_1 > 0$), temos $\|x\| \approx x_1$ e portanto $v_1 \approx 0$.

A derivada parcial de $v_1$ em relação a $x_1$ é:
$$\frac{\partial v_1}{\partial x_1} = 1 - \frac{x_1}{\|x\|} = 1 - \cos\theta,$$
onde $\theta$ é o ângulo entre $x$ e $e_1$. Quando $\theta \to 0$, $v_1 \to 0$ mas a derivada $\to 0$ também. O número de condicionamento **relativo** é:
$$\kappa_{\text{rel}}(v_1) = \left|\frac{\partial v_1}{\partial x_1} \cdot \frac{x_1}{v_1}\right| = \frac{(1 - x_1/\|x\|)\, x_1}{x_1 - \|x\|} = \frac{x_1/\|x\|(\|x\| - x_1)}{x_1 - \|x\|} = -\frac{x_1}{\|x\|}.$$

Para $x \approx \|x\|e_1$: $x_1/\|x\| \approx 1$, então $\kappa_{\text{rel}}(v_1) \approx 1$. Mas o problema é que $|v_1| \ll \|x\|$, então **erros absolutos** em $v_1$ são amplificados na divisão por $v_1$ ao calcular $\beta$.

### Fórmula alternativa para $v_1$

Inspirada na identidade $v_1 \cdot (x_1 + \|x\|) = x_1^2 - \|x\|^2 = -\|y\|^2$:
$$v_1 = x_1 - \|x\| = \frac{-\|y\|^2}{x_1 + \|x\|}.$$

Quando $x_1 > 0$, o denominador $x_1 + \|x\| \geq \|x\| > 0$ é estável. Esta fórmula **não sofre cancelamento** mesmo quando $x \approx \|x\|e_1$.

In [ ]:
using Printf

# Comparação de v₁ calculado pelas duas fórmulas conforme x → ||x||e₁
println("=== Condicionamento de v₁: fórmula direta vs. alternativa ===")
println("x = [1-ε, ε/√2, ε/√2]  para ε → 0")
println()
@printf "%-12s  %-20s  %-20s  %-15s\n" "ε" "v₁ direto" "v₁ alternativo" "erro relativo"
println("-"^72)

for exp in 1:15
    ε = 10.0^(-exp)
    x = [1 - ε, ε/sqrt(2), ε/sqrt(2)]
    nrmx = norm(x)
    
    # Fórmula direta: v₁ = x₁ - ||x||
    v1_direto = x[1] - nrmx
    
    # Fórmula alternativa: v₁ = -||y||² / (x₁ + ||x||)
    nrmy2 = x[2]^2 + x[3]^2
    v1_alt  = -nrmy2 / (x[1] + nrmx)
    
    err_rel = abs(v1_direto - v1_alt) / abs(v1_alt)
    @printf "1e-%-8d  %-20.6e  %-20.6e  %-15.6e\n" exp v1_direto v1_alt err_rel
end

In [ ]:
using Plots

# Gráfico do erro relativo de v₁ em função de ε
epsilons = [10.0^(-k) for k in 1:15]
erros = Float64[]

for ε in epsilons
    x    = [1 - ε, ε/sqrt(2), ε/sqrt(2)]
    nrmx = norm(x)
    v1_direto = x[1] - nrmx
    nrmy2 = x[2]^2 + x[3]^2
    v1_alt    = -nrmy2 / (x[1] + nrmx)
    push!(erros, abs(v1_direto - v1_alt) / abs(v1_alt))
end

plot(epsilons, erros,
     xscale=:log10, yscale=:log10,
     xlabel="ε  (x ≈ (1-ε)e₁)",
     ylabel="Erro relativo de v₁",
     title="Cancelamento em v₁ = x₁ - ||x|| quando x → e₁",
     label="|v₁ direto - v₁ alt| / |v₁ alt|",
     marker=:circle, lw=2, legend=:topleft)

# Linha de referência: eps Float64
hline!([eps(Float64)], linestyle=:dash, color=:red, label="eps(Float64) ≈ 2.2e-16")

O gráfico acima confirma que, à medida que $\varepsilon \to 0$ (i.e., $x \to e_1$), o erro relativo de $v_1$ calculado diretamente cresce até a ordem da precisão da máquina. A fórmula alternativa $v_1 = -\|y\|^2/(x_1 + \|x\|)$ não sofre esse problema.